# Quantum Circuit Design for Federated Quantum ML

This notebook demonstrates the implementation and testing of quantum components for the Distributed Federated Quantum Machine Learning Simulator.

## Contents

1. Setup and Installation
2. Basic Quantum Circuit Design
3. Data Encoding Strategies
4. Creating Variational Quantum Models
5. Training and Evaluation
6. Working with Noise Models
7. Performance Comparison

Let's get started!

## 1. Setup and Installation

First, let's make sure we have all the necessary packages installed:

In [ ]:
# Install required packages if not already installed
!pip install pennylane pennylane-qiskit qiskit qiskit-aer matplotlib scikit-learn torch numpy yaml pytest

In [ ]:
# Import necessary libraries
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
import torch
from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import yaml

# Add the project root to the path
# Adjust this path if needed to point to your project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import our quantum module
try:
    from quantum_federated_simulator.quantum.utils import load_config, save_circuit_diagram, set_random_seed
    from quantum_federated_simulator.quantum.circuits import create_basic_circuit, create_complex_circuit, create_custom_circuit
    from quantum_federated_simulator.quantum.encodings import angle_encoding, amplitude_encoding, basis_encoding, iqp_feature_map
    from quantum_federated_simulator.quantum.models import VariationalQuantumClassifier, QuantumNeuralNetwork, HybridQuantumModel
    print("Successfully imported quantum module")
except ImportError as e:
    print(f"Error importing quantum module: {e}")
    print("Please make sure the project structure is correct and all files are in place.")

# Set random seed for reproducibility
set_random_seed(42)

## 2. Basic Quantum Circuit Design

Let's explore the quantum circuits we've implemented:

In [ ]:
# Create a basic circuit
n_qubits = 4
n_layers = 2
basic_circuit = create_basic_circuit(n_qubits, n_layers)

# Create a device
dev = qml.device("default.qubit", wires=n_qubits)

# Create a QNode
basic_qnode = qml.QNode(basic_circuit, dev)

# Create random parameters and features
n_params = n_qubits * 3 * n_layers  # 3 rotation gates per qubit per layer
params = np.random.uniform(0, 2*np.pi, size=n_params)
features = np.random.uniform(-1, 1, size=n_qubits)

# Visualize the circuit
fig, ax = qml.draw_mpl(basic_qnode)(params, features)
plt.figure(figsize=(12, 6))
fig.set_size_inches(12, 6)
plt.tight_layout()
plt.title("Basic Quantum Circuit")
plt.show()

In [ ]:
# Create a complex circuit
complex_circuit = create_complex_circuit(n_qubits, n_layers)

# Create a QNode
complex_qnode = qml.QNode(complex_circuit, dev)

# Calculate parameters for complex circuit
rotation_params = n_qubits * 3 * n_layers
pairs_count = n_qubits + (n_qubits if n_qubits > 3 else 0)
entanglement_params = pairs_count * 2 * n_layers
complex_params = np.random.uniform(0, 2*np.pi, size=rotation_params + entanglement_params)

# Visualize the complex circuit
fig, ax = qml.draw_mpl(complex_qnode)(complex_params, features)
plt.figure(figsize=(14, 8))
fig.set_size_inches(14, 8)
plt.tight_layout()
plt.title("Complex Quantum Circuit")
plt.show()

## 3. Data Encoding Strategies

Let's examine different ways to encode classical data into quantum states:

In [ ]:
# Sample data
sample_data = np.array([0.5, -0.3, 0.7, -0.1])
wires = list(range(n_qubits))

# Define encoding circuits
def angle_encoding_circuit():
    angle_encoding(sample_data, wires)()
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

def amplitude_encoding_circuit():
    amplitude_encoding(sample_data, wires)()
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

def basis_encoding_circuit():
    # Convert to binary values for basis encoding
    binary_data = np.array([1 if val > 0 else 0 for val in sample_data])
    basis_encoding(binary_data, wires)()
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

def iqp_encoding_circuit():
    iqp_feature_map(sample_data, wires)()
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Create QNodes for each encoding
angle_qnode = qml.QNode(angle_encoding_circuit, dev)
amplitude_qnode = qml.QNode(amplitude_encoding_circuit, dev)
basis_qnode = qml.QNode(basis_encoding_circuit, dev)
iqp_qnode = qml.QNode(iqp_encoding_circuit, dev)

# Visualize each encoding
fig, axs = plt.subplots(2, 2, figsize=(16, 12))

# Angle encoding
angle_fig, _ = qml.draw_mpl(angle_qnode)()
axs[0, 0].set_title("Angle Encoding")
axs[0, 0].axis('off')
for item in angle_fig.get_children():
    if isinstance(item, plt.Axes):
        copied_item = angle_fig.get_children()[0].get_children()
        for subitem in copied_item:
            axs[0, 0].add_artist(subitem)

# Amplitude encoding
amp_fig, _ = qml.draw_mpl(amplitude_qnode)()
axs[0, 1].set_title("Amplitude Encoding")
axs[0, 1].axis('off')
for item in amp_fig.get_children():
    if isinstance(item, plt.Axes):
        copied_item = amp_fig.get_children()[0].get_children()
        for subitem in copied_item:
            axs[0, 1].add_artist(subitem)

# Basis encoding
basis_fig, _ = qml.draw_mpl(basis_qnode)()
axs[1, 0].set_title("Basis Encoding")
axs[1, 0].axis('off')
for item in basis_fig.get_children():
    if isinstance(item, plt.Axes):
        copied_item = basis_fig.get_children()[0].get_children()
        for subitem in copied_item:
            axs[1, 0].add_artist(subitem)

# IQP encoding
iqp_fig, _ = qml.draw_mpl(iqp_qnode)()
axs[1, 1].set_title("IQP Feature Map")
axs[1, 1].axis('off')
for item in iqp_fig.get_children():
    if isinstance(item, plt.Axes):
        copied_item = iqp_fig.get_children()[0].get_children()
        for subitem in copied_item:
            axs[1, 1].add_artist(subitem)

plt.tight_layout()
plt.show()

## 4. Creating Variational Quantum Models

Let's create and examine our quantum machine learning models:

In [ ]:
# Create a Variational Quantum Classifier
vqc = VariationalQuantumClassifier(
    n_qubits=4,
    n_layers=2,
    n_classes=2,
    circuit_type='basic',
    encoding_type='angle',
    device_name='default.qubit'
)

# Print model info
print(f"VQC Parameters: {vqc.n_params}")
print(f"VQC Circuit Type: {vqc.circuit_type}")
print(f"VQC Encoding Type: {vqc.encoding_type}")

# Create a Quantum Neural Network
qnn = QuantumNeuralNetwork(
    n_qubits=4,
    n_layers=2,
    input_size=4,
    output_size=2,
    circuit_type='complex',
    encoding_type='angle',
    pre_processing=True,
    post_processing=True
)

# Print QNN structure
print("\nQNN Structure:")
print(qnn)

## 5. Training and Evaluation

Let's train our VQC on a simple classification dataset:

In [ ]:
# Generate a simple binary classification dataset
X, y = make_classification(
    n_samples=100, 
    n_features=4,  # Match our n_qubits
    n_classes=2, 
    n_clusters_per_class=1,
    n_informative=2,
    random_state=42
)

# Scale features to [-1, 1] range for better quantum encoding
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.clip(X_scaled, -1, 1)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Create a new VQC for training
vqc_train = VariationalQuantumClassifier(
    n_qubits=4,
    n_layers=2,
    n_classes=2,
    circuit_type='basic',
    encoding_type='angle',
    device_name='default.qubit'
)

# Train the model
print("Training VQC model...")
loss_history = vqc_train.fit(
    X_train, y_train, 
    epochs=20,  # Using fewer epochs for demonstration
    batch_size=5, 
    learning_rate=0.1,
    verbose=True
)

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(loss_history)
plt.title('VQC Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
# Evaluate on test set
predictions = [vqc_train.predict(x) for x in X_test]
accuracy = np.mean(predictions == y_test)
print(f"Test accuracy: {accuracy:.4f}")

# Look at the first few predictions
print("\nSample predictions:")
for i in range(min(5, len(X_test))):
    print(f"True: {y_test[i]}, Predicted: {predictions[i]}")

Now let's train a PyTorch Quantum Neural Network:

In [ ]:
# Create tensor datasets
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create a QNN with simpler architecture for faster training
qnn_train = QuantumNeuralNetwork(
    n_qubits=4,
    n_layers=1,  # Using fewer layers for demonstration
    input_size=4,
    output_size=2,
    circuit_type='basic',
    encoding_type='angle',
    pre_processing=True,
    post_processing=True
)

# Setup training
optimizer = torch.optim.Adam(qnn_train.parameters(), lr=0.01)
loss_fn = torch.nn.CrossEntropyLoss()

# Training loop
n_epochs = 10  # Using fewer epochs for demonstration
batch_size = 5
train_losses = []

print("Training QNN model...")
for epoch in range(n_epochs):
    # Shuffle data
    indices = torch.randperm(len(X_train_tensor))
    X_train_shuffled = X_train_tensor[indices]
    y_train_shuffled = y_train_tensor[indices]
    
    # Process in batches
    epoch_loss = 0.0
    n_batches = len(X_train_tensor) // batch_size
    
    for i in range(n_batches):
        start_idx = i * batch_size
        end_idx = start_idx + batch_size
        
        X_batch = X_train_shuffled[start_idx:end_idx]
        y_batch = y_train_shuffled[start_idx:end_idx]
        
        # Forward pass
        outputs = qnn_train(X_batch)
        loss = loss_fn(outputs, y_batch)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    
    if epoch % 2 == 0 or epoch == n_epochs - 1:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.6f}")

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses)
plt.title('QNN Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
# Evaluate QNN
qnn_train.eval()
with torch.no_grad():
    outputs = qnn_train(X_test_tensor)
    _, predicted = torch.max(outputs, 1)
    
qnn_accuracy = (predicted == y_test_tensor).sum().item() / len(y_test_tensor)
print(f"QNN Test accuracy: {qnn_accuracy:.4f}")

# Look at the first few predictions
print("\nSample QNN predictions:")
for i in range(min(5, len(X_test))):
    print(f"True: {y_test[i]}, Predicted: {predicted[i].item()}")

## 6. Working with Noise Models

Let's explore how to simulate quantum noise in our models:

In [ ]:
# Create a noise config file
noise_config = {
    "noise_model": {
        "bit_flip": {
            "active": True,
            "probability": 0.01
        },
        "depolarizing": {
            "active": True,
            "probability": 0.005,
            "target": "gates"
        }
    },
    "readout_error": {
        "active": True,
        "probabilities": {
            "0|0": 0.98,
            "1|0": 0.02,
            "0|1": 0.03,
            "1|1": 0.97
        }
    }
}

# Save the noise config to a temporary file
noise_config_path = "temp_noise_config.yaml"
with open(noise_config_path, "w") as f:
    yaml.dump(noise_config, f)

# Create a noisy circuit using our basic circuit
basic_circuit_fn = create_basic_circuit(n_qubits=2, n_layers=1)
noise_model = noise_config["noise_model"]
noisy_circuit_fn = create_noisy_circuit(basic_circuit_fn, noise_model)

# Note: In a real implementation, we would use a device with noise capabilities
# For demonstration, we'll use the default simulator and explain what happens with noise

# Create a device
noise_dev = qml.device("default.qubit", wires=2)
noisy_qnode = qml.QNode(noisy_circuit_fn, noise_dev)

# Create random parameters and features
noise_params = np.random.uniform(0, 2*np.pi, size=2*3*1)  # 2 qubits * 3 rotations * 1 layer
noise_features = np.random.uniform(-1, 1, size=2)

# Execute the noisy circuit
print("Executing noisy circuit...")
result_noisy = noisy_qnode(noise_params, noise_features)
print(f"Noisy circuit output: {result_noisy}")
print("\nNote: This is a simplified demonstration of the concept of noisy circuits.")
print("In a real implementation, we would use a simulator with actual noise capabilities,")
print("such as 'qiskit.aer' with a custom noise model, or a hardware device with natural noise.")

# Clean up
if os.path.exists(noise_config_path):
    os.remove(noise_config_path)

## 7. Performance Comparison

Let's compare the performance of our quantum models with a classical model on the Iris dataset:

In [ ]:
# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Focus on binary classification (first two classes only)
binary_filter = y < 2
X_binary = X[binary_filter]
y_binary = y[binary_filter]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_binary)
X_scaled = np.clip(X_scaled, -1, 1)  # Clip to [-1, 1] range

# Reduce to 4 features (for 4 qubits)
X_scaled = X_scaled[:, :4]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_binary, test_size=0.3, random_state=42)

# Create models
# 1. Classical model (Logistic Regression)
from sklearn.linear_model import LogisticRegression
classical_model = LogisticRegression(random_state=42)

# 2. VQC model
vqc_iris = VariationalQuantumClassifier(
    n_qubits=4,
    n_layers=2,
    n_classes=2,
    circuit_type='basic',
    encoding_type='angle',
    device_name='default.qubit'
)

# Train classical model
print("Training classical model...")
classical_model.fit(X_train, y_train)

# Train VQC model
print("\nTraining VQC model...")
vqc_iris.fit(X_train, y_train, epochs=20, batch_size=5, learning_rate=0.05, verbose=True)

# Evaluate classical model
classical_predictions = classical_model.predict(X_test)
classical_accuracy = np.mean(classical_predictions == y_test)

# Evaluate VQC model
vqc_predictions = [vqc_iris.predict(x) for x in X_test]
vqc_accuracy = np.mean(vqc_predictions == y_test)

# Print results
print("\nPerformance Comparison:")
print(f"Classical model accuracy: {classical_accuracy:.4f}")
print(f"VQC model accuracy: {vqc_accuracy:.4f}")

# Visualize results
plt.figure(figsize=(10, 6))
plt.bar(['Classical Model', 'Quantum VQC'], [classical_accuracy, vqc_accuracy], color=['blue', 'purple'])
plt.ylim(0, 1.0)
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, v in enumerate([classical_accuracy, vqc_accuracy]):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center')

plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we've explored the quantum components of our Distributed Federated Quantum Machine Learning Simulator:

1. We implemented and visualized various quantum circuit architectures
2. We explored different data encoding strategies
3. We built and trained quantum machine learning models
4. We compared the performance of quantum and classical models

These quantum components form the foundation of our federated quantum learning system. The next steps would be to integrate these quantum components with the federated learning framework to enable distributed quantum learning across multiple nodes.

Some key insights:
- Quantum circuits can be designed with different levels of complexity and expressivity
- Data encoding is a critical step in quantum machine learning
- Quantum models can achieve competitive performance with classical models on certain tasks
- The presence of noise can significantly affect quantum model performance

In future notebooks, we'll explore the federated learning and distributed computing aspects of our system.